# OKF Exploration

This notebook investigates the Open Knowledge Framework (OKF) design and its local integrations.
We will review folder structure, ingest patterns, knowledge graph construction, and how agent workflows can leverage indexed content.

## Environment Setup

In [7]:
# Safe magic usage so this cell can run as a script or in a notebook
try:
    ip = get_ipython()
except NameError:
    ip = None

if ip is not None:
    ip.run_line_magic('load_ext', 'autoreload')
    ip.run_line_magic('autoreload', '2')

import os
from pathlib import Path
import sys
import sqlite3
import pandas as pd

repo_root = Path.cwd().parent.resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

db_path = repo_root / "data" / "ledger.db"
okf_path = repo_root / "knowledge" / ".okf"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Empirical Data Extraction

In [8]:
# Connect to ledger and extract raw truth
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT cloud_path, nextcloud_path, valid_from FROM file_ledger WHERE is_active = 1", conn)

# Feature Extraction
df['filename'] = df['nextcloud_path'].apply(lambda x: os.path.basename(x))
df['extension'] = df['filename'].apply(lambda x: os.path.splitext(x)[1].lower())
df['parent_directory'] = df['nextcloud_path'].apply(lambda x: os.path.dirname(x))
df['depth'] = df['nextcloud_path'].apply(lambda x: len([p for p in x.split('/') if p]) - 1)
df['top_level'] = df['nextcloud_path'].apply(lambda x: x.split('/')[1] if len(x.split('/')) > 1 else 'root')

print(f"Total Files: {len(df)}")

# Display Extension Frequency
print("\n--- Extension Distribution ---")
display(df['extension'].value_counts().head(10))

# Display Top Level Directory Frequency
print("\n--- Top Level Directories ---")
display(df['top_level'].value_counts())

Total Files: 204

--- Extension Distribution ---


extension
.pdf    132
.mp3     63
.mkv      7
.mp4      2
Name: count, dtype: int64


--- Top Level Directories ---


top_level
Documents    204
Name: count, dtype: int64

In [9]:
conn.create_function("REVERSE", 1, lambda text: text[::-1] if text else "")

query = """
    SELECT 
        SUBSTR(nextcloud_path, 1, LENGTH(nextcloud_path) - INSTR(REVERSE(nextcloud_path), '/')) AS folder_path, 
        COUNT(id) as file_count
    FROM file_ledger
    GROUP BY folder_path
    ORDER BY folder_path ASC;
"""

# Creates a DataFrame directly from the SQL output
df = pd.read_sql_query(query, conn)
df

                                         folder_path  file_count
0  /Documents/GDriveIngestionTest/My Documents/Jo...         132
1  /Documents/GDriveIngestionTest/My Music/Artcel...           6
2  /Documents/GDriveIngestionTest/My Music/Artcel...          10
3  /Documents/GDriveIngestionTest/My Music/Artcel...           9
4  /Documents/GDriveIngestionTest/My Music/Aurtho...           1
5  /Documents/GDriveIngestionTest/My Music/Aurtho...          12
6  /Documents/GDriveIngestionTest/My Music/Aurtho...          15
7  /Documents/GDriveIngestionTest/My Music/Aurtho...          12
8  /Documents/GDriveIngestionTest/My Videos/TV Se...           7

## Generate OKF Scaffold

In [12]:
from src.okf.builder import OKFBuilder

# Deterministically build the OKF Markdown files from the ledger
builder = OKFBuilder(db_path=db_path, okf_dir=okf_path)
builder.build()

✅ OKF Knowledge Base built at /home/coder/projects/agentic-nas-workflow/knowledge/.okf
